# Minion of Mordred Role
The Minion of Mordred is an Evil player with no extra powers beyond being on the Evil team. You know who your evil teammates are (including the Assassin). You and the Assassin can see each other. Additionally, Merlin is able to see the Minion of Mordred in this variant.
### During The Game
Your responsibilities are the same as other evil players: argue, vote on teams, propose teams when it is your turn, and try to steer missions toward failing without revealing that you are evil.
If you are on a mission, choose to pass or fail the mission as appropriate for the team's strategy.
### Role's Code
Provide a role prompt that explains your role, the team you are on, and who the other evil players are. You do NOT have any assassination or special endgame actions.
Every turn, the environment will provide context such as the current round, past votes, mission outcomes, chat history, etc.
Maintain simple memory to track suspicious players or noteworthy events.
### End to End
1. Environment assigns the roles to everyone
2. Minion of Mordred gets their role and knows who the other evil people are (including the Assassin)
3. Game start
4. Discussion Phase where the minion gets information about the current state and generates a discussion
5. Voting Phase where the minion votes on proposed teams
6. Mission Phase if the minion is on the mission team, choose pass or fail
7. Repeat the process from 4-6
8. There is no assassination action for the minion; endgame decisions (like Assassin's assassination) are handled by the Assassin role.


In [1]:
from __future__ import annotations

from dataclasses import dataclass, field
from enum import Enum
from typing import Any, Dict, List, Optional, Protocol, Tuple
import json
import re
import random

random.seed(0)

In [ ]:
class MissionResult(str, Enum):
    SUCCESS = "success"
    FAIL = "fail"

# constrain the actions
class VoteDecision(str, Enum):
    APPROVE = "approve"
    REJECT = "reject"


class MissionAction(str, Enum):
    PASS = "pass"
    FAIL = "fail"


@dataclass
class Proposal:
    round_idx: int
    proposal_idx: int
    proposer: str
    team: List[str]
    votes: Dict[str, VoteDecision] = field(default_factory=dict)
    approved: Optional[bool] = None


@dataclass
class Mission:
    round_idx: int
    team: List[str]
    outcome: Optional[MissionResult] = None
    fail_count: Optional[int] = None

# the environment 
# gives the information at decision time
@dataclass
class GameState:
    players: List[str]
    me: str
    round_idx: int
    proposal_idx: int
    num_successes: int
    num_fails: int
    current_proposer: str
    current_team: List[str]
    chat_log: List[Tuple[str, str]] = field(default_factory=list)
    proposals: List[Proposal] = field(default_factory=list)
    missions: List[Mission] = field(default_factory=list)
    extra: Dict[str, Any] = field(default_factory=dict)

In [ ]:
# we just plug in our stuff and then it generates
class LLMBackend(Protocol):
    def generate(self, *, system: str, user: str, max_tokens: int = 400) -> str:
        ...




In [ ]:
# basically an edge case checker
# will try to fix if the LLM tries to respond in json
def _extract_json_object(text: str) -> Optional[dict]:
    text = text.strip()
    if not text:
        return None

    try:
        obj = json.loads(text)
        if isinstance(obj, dict):
            return obj
    except Exception:
        pass

    match = re.search(r"\{.*\}", text, flags=re.DOTALL)
    if not match:
        return None

    try:
        return json.loads(match.group(0))
    except Exception:
        return None


def require_fields(obj: dict, fields: List[str]) -> bool:
    return all(f in obj for f in fields)

In [ ]:
# main prompt
def build_minion_system_prompt(me: str, evil_team: List[str]) -> str:
    return (
        "You are playing The Resistance: Avalon.\n"
        "Your role: MINION OF MORDRED (EVIL).\n\n"
        f"You are: {me}\n"
        f"Your known Evil teammates (including you): {', '.join(evil_team)}\n\n"
        "Goals:\n"
        "1) Make missions fail.\n"
        "2) Avoid revealing you are Evil.\n\n"
        "When asked for actions, output ONLY valid JSON.\n"
    )


def summarize_recent_chat(chat_log: List[Tuple[str, str]], max_turns: int = 12) -> str:
    recent = chat_log[-max_turns:]
    if not recent:
        return "(no chat yet)"
    return "\n".join([f"{speaker}: {msg}" for speaker, msg in recent])

# basically the thing that gives context
def build_state_block(state: GameState) -> str:
    return (
        f"Players: {', '.join(state.players)}\n"
        f"Round: {state.round_idx} | Proposal: {state.proposal_idx}\n"
        f"Score: GoodSuccesses={state.num_successes}, EvilFails={state.num_fails}\n"
        f"Current proposer: {state.current_proposer}\n"
        f"Current proposed team: {state.current_team}\n\n"
        "Recent chat:\n"
        f"{summarize_recent_chat(state.chat_log)}\n"
    )

In [ ]:
# need to upgrade this
# the internal dialogues (might be able to make this more general for every role)
@dataclass
class MinionMemory:
    notes: List[str] = field(default_factory=list)
    suspicion: Dict[str, float] = field(default_factory=dict)
    def bump(self, player: str, delta: float) -> None:
        self.suspicion[player] = self.suspicion.get(player, 0.0) + delta
    def top_suspects(self, k: int = 2):
        return sorted(self.suspicion.items(), key=lambda x: x[1], reverse=True)[:k]

In [ ]:
# things it can do
class MinionAgent:
    def __init__(self, *, llm: LLMBackend, me: str, evil_team: List[str]):
        self.llm = llm
        self.me = me
        self.evil_team = evil_team
        self.system = build_minion_system_prompt(me, evil_team)
        self.memory = MinionMemory()

    def speak(self, state: GameState) -> str:
        user = (
            '{"type":"speech","message":string,"intent":string}\n\n'
            + build_state_block(state)
            + "\nTask: Produce 1-3 sentences for discussion."
        )
        raw = self.llm.generate(system=self.system, user=user)
        obj = _extract_json_object(raw) or {}
        return obj.get("message", "Let’s reconsider the logic behind this team.")

    def vote_on_team(self, state: GameState, team: List[str]) -> VoteDecision:
        user = (
            '{"type":"vote","decision":"approve|reject","reason":string}\n\n'
            + build_state_block(state)
            + f"\nVote on team: {team}"
        )
        raw = self.llm.generate(system=self.system, user=user)
        obj = _extract_json_object(raw) or {}
        decision = obj.get("decision", "").lower()
        if decision == "approve":
            return VoteDecision.APPROVE
        if decision == "reject":
            return VoteDecision.REJECT
        return VoteDecision.APPROVE if self.me in team else VoteDecision.REJECT

    def mission_action(self, state: GameState) -> MissionAction:
        user = (
            '{"type":"mission","action":"pass|fail","reason":string}\n\n'
            + build_state_block(state)
        )
        raw = self.llm.generate(system=self.system, user=user)
        obj = _extract_json_object(raw) or {}
        action = obj.get("action", "").lower()
        if action == "fail":
            return MissionAction.FAIL
        if action == "pass":
            return MissionAction.PASS
        return MissionAction.FAIL if state.num_successes >= 2 else MissionAction.PASS
